# 04 — CRSP Monthly Returns Panel

**Goal:** Construct a firm-month panel of stock returns for the S&P 500 universe.
This is the dependent variable for the asset-pricing tests (H1, H2a).

**Input:**
- `data/sp500_universe_with_gvkey.parquet` (717 unique gvkeys)
- `crsp.msf` — CRSP monthly stock file (~7M rows globally)
- `crsp.ccmxpf_linktable` — CRSP-Compustat bridge (links gvkey to permno)

**Output:** `data/returns_panel.parquet`
- One row per (gvkey, permno, date)
- Monthly returns (with and without dividends), prices, shares outstanding,
  market equity
- 2012–2024 (Jan 2012 through Dec 2024) — extending one year before the
  sample window to support lagged regressors

**Linking approach:**
- Link gvkey to permno via `ccmxpf_linktable`, restricted to primary links
  (`linkprim IN ('P', 'C')`) and link types `LU`, `LC` and `LS`
- **Revision 2026-06-10:** `LS` ("link valid for this security only") added —
  Eaton Corp plc and Dayforce carry their sole primary link as LS and were
  silently dropped under the previous `LU`/`LC`-only filter; no other universe
  firm has any LS link, so the addition affects exactly these two firms
- For each (gvkey, month), retain only the permno whose link spell covers that month
- Expect 1:1 (gvkey, month) → permno mapping in nearly all cases

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import wrds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PROCESSED = Path.home() / "thesis" / "data"

universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe_with_gvkey.parquet")
target_gvkeys = sorted(universe["gvkey"].dropna().unique())
print(f"Target gvkeys: {len(target_gvkeys)}")

Target gvkeys: 717


In [2]:
# Read the WRDS username from ~/.pgpass (4th field) so the notebook runs headlessly
WRDS_USERNAME = next(
    line.split(":")[3]
    for line in (Path.home() / ".pgpass").read_text().splitlines()
    if "wrds" in line
)
db = wrds.Connection(wrds_username=WRDS_USERNAME)

Loading library list...


Done


In [3]:
gvkey_list_sql = ",".join([f"'{g}'" for g in target_gvkeys])

# linktype: LC (confirmed) and LU (unconfirmed) primary links, plus LS
# ("link valid for this security only"). FIX 2026-06-10: exactly two universe
# firms carry their sole primary link as LS — Eaton Corp plc (004199, permno
# 11762, linkprim P since 1962) and Dayforce Inc (023546, permno 17700, P since
# its 2018 IPO). Excluding LS silently dropped both firms from the returns
# panel; no other universe gvkey has any LS link, so adding it changes nothing
# else (verified against crsp.ccmxpf_linktable on 2026-06-10).
link_query = f"""
    SELECT  gvkey,
            lpermno AS permno,
            linkprim,
            linktype,
            linkdt,
            linkenddt
    FROM    crsp.ccmxpf_linktable
    WHERE   gvkey IN ({gvkey_list_sql})
      AND   linktype IN ('LU', 'LC', 'LS')
      AND   linkprim IN ('P', 'C')
"""

links = db.raw_sql(link_query)

# Coerce dates
links["linkdt"] = pd.to_datetime(links["linkdt"])
# linkenddt is NaN for "still active" — replace with a future sentinel for the join
links["linkenddt"] = pd.to_datetime(links["linkenddt"]).fillna(pd.Timestamp("2099-12-31"))

print(f"Link rows: {len(links)}")
print(f"Unique gvkeys with links: {links['gvkey'].nunique()}")
print(f"Unique permnos: {links['permno'].nunique()}")
print()
print("Gvkeys without any link:")
no_link = set(target_gvkeys) - set(links["gvkey"])
print(f"  Count: {len(no_link)}")
print(f"  Sample: {sorted(no_link)[:10] if no_link else 'none'}")
print()

# Some firms have multiple links across time — check
links_per_gvkey = links.groupby("gvkey").size()
print(f"Gvkeys with multiple link rows: {(links_per_gvkey > 1).sum()}")
print(f"Max links for one gvkey: {links_per_gvkey.max()}")

Link rows: 945
Unique gvkeys with links: 717
Unique permnos: 815

Gvkeys without any link:
  Count: 0
  Sample: none

Gvkeys with multiple link rows: 184
Max links for one gvkey: 5


In [4]:
# Identify any firms without CRSP links (should be NONE after the LS fix —
# previously Eaton 004199 and Dayforce 023546 were dropped here)
no_link_gvkeys = [g for g in target_gvkeys if g not in set(links["gvkey"])]
print(f"Firms without CRSP links: {no_link_gvkeys if no_link_gvkeys else 'none'}")

if no_link_gvkeys:
    no_link_sql = ",".join([f"'{g}'" for g in no_link_gvkeys])
    diag = db.raw_sql(f"""
        SELECT gvkey, conm, fic, ipodate, dldte
        FROM   comp.company
        WHERE  gvkey IN ({no_link_sql})
    """)
    print(diag)

Firms without CRSP links: none


In [5]:
permno_list = links["permno"].astype(int).unique().tolist()
permno_list_sql = ",".join([str(p) for p in permno_list])

returns_query = f"""
    SELECT  permno,
            date,
            ret,
            retx AS ret_ex_dividend,
            prc,
            shrout
    FROM    crsp.msf
    WHERE   permno IN ({permno_list_sql})
      AND   date BETWEEN '2012-01-01' AND '2024-12-31'
"""

print("Running query — may take 1-5 minutes...")
returns = db.raw_sql(returns_query)
print(f"Pulled {len(returns):,} firm-month rows")

Running query — may take 1-5 minutes...


Pulled 100,284 firm-month rows


In [6]:
returns["date"] = pd.to_datetime(returns["date"])
returns["year"] = returns["date"].dt.year
returns["month"] = returns["date"].dt.month

print(f"Shape: {returns.shape}")
print(f"Date range: {returns['date'].min().date()} to {returns['date'].max().date()}")
print(f"Unique permnos: {returns['permno'].nunique()}")
print()

print("Returns per year:")
print(returns.groupby("year").size())
print()

print("Return distribution (`ret`):")
print(returns["ret"].describe())
print()

print("Missing rates:")
print(returns.isna().mean().sort_values(ascending=False).round(3))
print()

print("Negative price count (CRSP convention for missing close):")
print((returns["prc"] < 0).sum())
print()

print("Sample rows:")
print(returns.head())

Shape: (100284, 8)
Date range: 2012-01-31 to 2024-12-31
Unique permnos: 750

Returns per year:
year
2012    8015
2013    8112
2014    8129
2015    8128
2016    7938
2017    7825
2018    7758
2019    7651
2020    7500
2021    7412
2022    7350
2023    7257
2024    7209
dtype: int64

Return distribution (`ret`):
count     99941.0
mean     0.013363
std      0.108338
min     -0.886269
25%     -0.036079
50%       0.01273
75%      0.059991
max      16.25053
Name: ret, dtype: Float64

Missing rates:
ret                0.003
ret_ex_dividend    0.003
prc                0.003
shrout             0.001
permno             0.000
date               0.000
year               0.000
month              0.000
dtype: float64

Negative price count (CRSP convention for missing close):
180

Sample rows:
   permno       date       ret  ret_ex_dividend     prc     shrout  year  month
0   10104 2012-01-31  0.102144         0.099805   28.21  5025837.0  2012      1
1   10104 2012-02-29  0.037044         0.037044  2

In [7]:
# Merge returns to links on permno
merged = returns.merge(links, on="permno", how="left")

# Keep rows where the return date falls within the link's effective spell
mask = (
    (merged["date"] >= merged["linkdt"]) &
    (merged["date"] <= merged["linkenddt"])
)
returns_with_gvkey = merged[mask].copy()

print(f"Returns rows before join: {len(returns):,}")
print(f"Returns rows after join: {len(returns_with_gvkey):,}")
print()

# Check for duplicates (one firm-month should have exactly one gvkey)
dup_check = returns_with_gvkey.groupby(["permno", "date"])["gvkey"].nunique()
multi_gvkey = (dup_check > 1).sum()
print(f"(permno, date) pairs with multiple gvkey assignments: {multi_gvkey}")

# And check the reverse — one gvkey-month should have exactly one permno
dup_check_2 = returns_with_gvkey.groupby(["gvkey", "date"])["permno"].nunique()
multi_permno = (dup_check_2 > 1).sum()
print(f"(gvkey, date) pairs with multiple permno assignments: {multi_permno}")
print()

# Drop the linking columns we don't need anymore
returns_with_gvkey = returns_with_gvkey.drop(
    columns=["linkprim", "linktype", "linkdt", "linkenddt"]
)

# Final shape
print(f"Unique gvkeys with returns: {returns_with_gvkey['gvkey'].nunique()}")
print(f"Unique permnos with returns: {returns_with_gvkey['permno'].nunique()}")
print(f"Rows per year:")
print(returns_with_gvkey.groupby("year").size())

Returns rows before join: 100,284
Returns rows after join: 98,978

(permno, date) pairs with multiple gvkey assignments: 0
(gvkey, date) pairs with multiple permno assignments: 0

Unique gvkeys with returns: 717
Unique permnos with returns: 745
Rows per year:
year
2012    7883
2013    7987
2014    8009
2015    7986
2016    7799
2017    7722
2018    7666
2019    7569
2020    7412
2021    7331
2022    7272
2023    7195
2024    7147
dtype: int64


In [8]:
# Take absolute value of price — CRSP encodes bid-ask midpoint as negative
returns_with_gvkey["prc_abs"] = returns_with_gvkey["prc"].abs()

# Market equity = price × shares outstanding
# shrout is in thousands, so multiply by 1000 for actual share count, then divide
# by 1,000,000 to get market cap in millions (standard reporting unit)
returns_with_gvkey["me"] = (
    returns_with_gvkey["prc_abs"] * returns_with_gvkey["shrout"] / 1000
)

# Reorder columns and finalize
final = returns_with_gvkey[[
    "gvkey", "permno", "date", "year", "month",
    "ret", "ret_ex_dividend", "prc", "shrout", "me",
]].sort_values(["gvkey", "date"]).reset_index(drop=True)

print(f"Final shape: {final.shape}")
print()

print("Market equity distribution (millions of USD):")
print(final["me"].describe().round(1))
print()

# Sanity check: largest firms by market cap should be recognizable
print("Top 10 firm-months by market cap (should be obvious mega-caps):")
top = final.nlargest(10, "me")[["gvkey", "permno", "date", "me"]]
print(top)
print()

# Cross-reference top gvkeys against the universe to see firm names
top_gvkeys = top["gvkey"].unique().tolist()
top_gvkeys_sql = ",".join([f"'{g}'" for g in top_gvkeys])
names = db.raw_sql(f"""
    SELECT gvkey, conm
    FROM   comp.company
    WHERE  gvkey IN ({top_gvkeys_sql})
""")
print("Top firms identified:")
print(names)

Final shape: (98978, 10)

Market equity distribution (millions of USD):
count      98886.0
mean       42419.9
std       119840.6
min           15.5
25%         8124.9
50%        16292.4
75%        37028.7
max      3785304.4
Name: me, dtype: Float64

Top 10 firm-months by market cap (should be obvious mega-caps):
        gvkey  permno       date             me
3051   001690   14593 2024-12-31  3785304.39566
3050   001690   14593 2024-11-29  3587438.27259
3048   001690   14593 2024-09-30    3522211.138
3047   001690   14593 2024-08-30    3481747.373
3049   001690   14593 2024-10-31  3414815.57393
78542  117768   86580 2024-11-29      3385742.5
3046   001690   14593 2024-07-31  3376534.74496
45369  012141   10107 2024-06-28  3322626.37434
78543  117768   86580 2024-12-31   3288761.8551
78541  117768   86580 2024-10-31  3253681.83492



Top firms identified:
    gvkey            conm
0  001690       APPLE INC
1  012141  MICROSOFT CORP
2  117768     NVIDIA CORP


In [9]:
output_path = DATA_PROCESSED / "returns_panel.parquet"
final.to_parquet(output_path, index=False)

check = pd.read_parquet(output_path)
print(f"Saved {len(check):,} rows to {output_path.name}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print()

# Coverage check: how many universe firm-years have at least one return observation?
universe = pd.read_parquet(DATA_PROCESSED / "sp500_universe_with_gvkey.parquet")
universe["gvkey"] = universe["gvkey"].astype(str)
check["gvkey"] = check["gvkey"].astype(str)

# Aggregate returns to firm-year (does each firm have returns each year?)
returns_yearly = (
    check.groupby(["gvkey", "year"])
    .size()
    .reset_index(name="months_with_returns")
)
returns_yearly["year"] = returns_yearly["year"].astype(int)

merged_check = universe.merge(returns_yearly, on=["gvkey", "year"], how="left")
print(f"Universe firm-year rows: {len(merged_check):,}")
print(f"  With ≥1 month of returns: {merged_check['months_with_returns'].notna().sum():,}")
print(f"  With 12 months of returns: {(merged_check['months_with_returns'] == 12).sum():,}")
print(f"  Coverage rate (any returns): {merged_check['months_with_returns'].notna().mean():.1%}")
print()

print("Universe coverage by year:")
coverage = merged_check.groupby("year").agg(
    universe_firms=("gvkey", "nunique"),
    with_full_year=("months_with_returns", lambda x: (x == 12).sum()),
).assign(full_year_pct=lambda d: d["with_full_year"] / d["universe_firms"])
print(coverage.round(3))

Saved 98,978 rows to returns_panel.parquet
File size: 3394.8 KB

Universe firm-year rows: 6,535
  With ≥1 month of returns: 6,524
  With 12 months of returns: 6,491
  Coverage rate (any returns): 99.8%

Universe coverage by year:
      universe_firms  with_full_year  full_year_pct
year                                               
2012             497             491          0.988
2013             497             492          0.990
2014             496             497          1.002
2015             497             494          0.994
2016             500             502          1.004
2017             499             502          1.006
2018             500             505          1.010
2019             500             500          1.000
2020             500             502          1.004
2021             500             504          1.008
2022             500             502          1.004
2023             500             501          1.002
2024             500             499      